# 1. Data acquisition, exploration, and preprocessing

**Research question:** Do Random Forest and XGBoost improve on simple prediction and portfolio baselines?

Use 2015–2021 for training and 2022–2023 for validation. Preserve 2024–2025 for the assignment test and **2026 as a separate competition holdout**. No 2026 performance is explored here.

Run the three notebooks in numerical order, each in a fresh kernel. Install `requirements.txt` and select that environment. Shared, tested implementations are in `stockml/`; retain that folder with the notebooks. Normal runs download real data. The automated smoke runner explicitly uses deterministic synthetic data and reduced models.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/terminal34-datascience-bootcamp/stock-market-ml/blob/main/01_data_preprocessing.ipynb)

## Run locally or in Google Colab

In Colab, select a **Python 3 CPU runtime**, then run the setup cell below and authorize its Google Drive mount. It loads the shared code from GitHub and installs the runtime dependencies. Use the same `EXPERIMENT_NAME` and `COLAB_SMOKE_TEST` values in notebooks **1 → 2 → 3**; each can run in a separate session. Your datasets, models, predictions, and frozen selection are saved directly under `My Drive/stock-market-ml/<experiment>/real/` (or `smoke/`), so they survive runtime resets. Save a notebook copy to Drive if you want to keep your notes and cell outputs too.

**Private GitHub repository:** add a `GITHUB_TOKEN` in Colab's Secrets panel (key icon), using a fine-grained token with **Contents: Read-only** access to this repository, and enable notebook access in each notebook. Organization approval may be required. Do not paste the token into a code cell. Setup passes it only to Git subprocesses and does not save it in artifacts or Git remote URLs. If Colab cannot open the private GitHub link, download the `.ipynb` from GitHub and upload it to Colab; the same secret enables the code download.

The first notebook records a Git commit and exact package versions; later notebooks reuse them. If setup asks for a runtime restart after installing packages, restart the session and rerun setup. Run one notebook at a time for a given experiment. A missing-handoff error means you need to finish the preceding notebook with the same settings. On a local machine, setup preserves your environment and artifact path; install `requirements.txt` first.

Changing the experiment folder after seeing 2026 results does not restore holdout independence. To try this Colab port without revisiting the real holdout, set `COLAB_SMOKE_TEST=True` in all three notebooks. Your existing local artifacts are not uploaded automatically.

In [ ]:
# Use the SAME values in all three Colab notebooks.
EXPERIMENT_NAME = 'competition-v1'
COLAB_SMOKE_TEST = False  # True = synthetic data and small models, stored separately.

"""Standard-library-only bootstrap, embedded into each notebook before third-party imports."""
from pathlib import Path
import base64
import importlib.metadata
import importlib.util
import json
import os
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/terminal34-datascience-bootcamp/stock-market-ml.git'
RUNTIME_PACKAGES = {
    'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
    'scikit-learn': 'sklearn', 'xgboost': 'xgboost', 'yfinance': 'yfinance',
    'pyarrow': 'pyarrow', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'joblib': 'joblib',
}


def in_colab():
    try:
        return importlib.util.find_spec('google.colab') is not None
    except (ImportError, ValueError):
        return False


def require_handoff(artifacts, notebook_number):
    required = {
        1: [],
        2: ['data/metadata.json'],
        3: ['frozen_experiment.json', 'competition_evaluation.json',
            'predictions/assignment.parquet', 'predictions/competition.parquet'],
    }
    if notebook_number not in required:
        raise ValueError('Notebook number must be 1, 2, or 3')
    missing = [name for name in required[notebook_number] if not (artifacts / name).is_file()]
    if missing:
        raise RuntimeError(
            f'Missing handoff files in {artifacts}: {missing}. '
            f'Run Notebook {notebook_number - 1} first, with the same EXPERIMENT_NAME and smoke setting.'
        )


def github_git_environment(token=None):
    # Headers live only in child-process environments, never URLs, arguments, or .git/config.
    environment = {key: value for key, value in os.environ.items()
                   if not key.startswith('GIT_TRACE') and key != 'GIT_CURL_VERBOSE'}
    environment['GIT_TERMINAL_PROMPT'] = '0'
    if token:
        encoded = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        environment.update({'GIT_CONFIG_COUNT': '1',
                            'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
                            'GIT_CONFIG_VALUE_0': f'Authorization: Basic {encoded}'})
    return environment


def prepare_colab_checkout(artifacts, project, repository_url=REPOSITORY_URL, token=None):
    """Use one source revision and resolved package set for all three sessions."""
    artifacts, project = Path(artifacts), Path(project)
    artifacts.mkdir(parents=True, exist_ok=True)
    manifest_path = artifacts / 'colab_environment.json'
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
    python_version = f'{sys.version_info.major}.{sys.version_info.minor}'
    if manifest and (manifest['repository_url'] != repository_url or manifest['python'] != python_version):
        raise RuntimeError('This experiment belongs to a different repository or Python version. '
                           'Use the recorded runtime version to reuse its models.')
    git_environment = github_git_environment(token)
    if not project.exists():
        try:
            subprocess.check_call(['git', 'clone', '--quiet', repository_url, str(project)], env=git_environment)
        except subprocess.CalledProcessError as error:
            raise RuntimeError('Cannot clone the repository. For this private repo, add a GITHUB_TOKEN '
                               'Colab secret with Contents: Read access, enable notebook access, and rerun setup.') from error
    if not (project / '.git').is_dir():
        raise RuntimeError(f'{project} exists but is not a Git checkout; choose a fresh Colab session.')
    git = ['git', '-C', str(project)]
    origin = subprocess.check_output(git + ['remote', 'get-url', 'origin'], text=True).strip()
    if origin != repository_url:
        raise RuntimeError(f'Refusing to use an unrelated checkout at {project}')
    if subprocess.check_output(git + ['status', '--porcelain'], text=True).strip():
        raise RuntimeError('The Colab checkout has local edits. Save them before using a fresh runtime; '
                           'setup will not overwrite your work.')
    ref = manifest['commit'] if manifest else 'main'
    subprocess.check_call(git + ['fetch', '--quiet', 'origin', ref], env=git_environment)
    commit = subprocess.check_output(git + ['rev-parse', 'FETCH_HEAD'], text=True).strip()
    subprocess.check_call(git + ['checkout', '--quiet', '--detach', commit])
    requirements = project / 'requirements-colab.txt'
    if not requirements.is_file():
        raise RuntimeError('This GitHub revision does not include Colab support. Publish the updated project first.')
    if manifest:
        requirements = artifacts / 'colab-runtime-requirements.txt'
        requirements.write_text(''.join(f'{name}=={version}\n' for name, version in manifest['packages'].items()))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(requirements)])
    resolved = {name: importlib.metadata.version(name) for name in RUNTIME_PACKAGES}
    if manifest and resolved != manifest['packages']:
        raise RuntimeError('Installed packages do not match the saved Colab environment')
    if manifest is None:
        manifest = {'repository_url': repository_url, 'commit': commit,
                    'python': python_version, 'packages': resolved}
        temporary = manifest_path.with_suffix('.json.tmp')
        temporary.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n')
        temporary.replace(manifest_path)
        (artifacts / 'colab-runtime-requirements.txt').write_text(
            ''.join(f'{name}=={version}\n' for name, version in resolved.items()))
    changed = [name for name, module in RUNTIME_PACKAGES.items()
               if module in sys.modules and getattr(sys.modules[module], '__version__', resolved[name]) != resolved[name]]
    if changed:
        raise RuntimeError(f'Installed versions changed for already-imported packages: {changed}. '
                           'Choose Runtime > Restart session, then run this setup cell again.')
    print('Pinned GitHub revision:', commit)
    return project


def setup_notebook(notebook_number, experiment_name='competition-v1', colab_smoke=False):
    if in_colab():
        # Authorization is performed by Colab's own Drive prompt, in the user's runtime.
        from google.colab import drive, userdata
        if not experiment_name or experiment_name in ['.', '..'] or any(c in experiment_name for c in '/\\'):
            raise ValueError('EXPERIMENT_NAME must be one folder name')
        drive.mount('/content/drive')
        artifacts = Path('/content/drive/MyDrive/stock-market-ml') / experiment_name / ('smoke' if colab_smoke else 'real')
        require_handoff(artifacts, notebook_number)
        try:
            token = userdata.get('GITHUB_TOKEN')
        except userdata.SecretNotFoundError:
            token = None  # Public forks need no token; private-repo errors explain the setup.
        except userdata.NotebookAccessError as error:
            raise RuntimeError('Enable notebook access for GITHUB_TOKEN in the Colab Secrets panel.') from error
        project = prepare_colab_checkout(artifacts, Path('/content/stock-market-ml'), token=token)
        os.environ['STOCK_ML_ARTIFACTS'] = str(artifacts)
        os.environ['STOCK_ML_SMOKE'] = '1' if colab_smoke else '0'
        os.chdir(project)
    else:
        project = Path.cwd()
        if not (project / 'stockml').is_dir():
            raise RuntimeError('For local execution, open the notebook from the repository root.')
        artifacts = Path(os.environ.get('STOCK_ML_ARTIFACTS', 'artifacts')).resolve()
        require_handoff(artifacts, notebook_number)
    sys.path.insert(0, str(project))
    os.environ.setdefault('MPLCONFIGDIR', str(project / '.cache' / 'matplotlib'))
    Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
    print('Artifact handoff folder:', artifacts)
    return project

PROJECT = setup_notebook(1, EXPERIMENT_NAME, COLAB_SMOKE_TEST)

import pandas as pd
from IPython.display import display, Markdown
from stockml.common import root, smoke, load_json

pd.set_option('display.max_columns', 20)
print('Artifacts:', root())
print('SYNTHETIC SMOKE TEST — NOT FINANCIAL RESULTS' if smoke() else 'REAL-DATA EXPERIMENT')

## Acquire and lock a reproducible snapshot

`AS_OF=None` uses the latest available session through yesterday in New York, capped at December 2026. The current day is conservatively excluded. A cached snapshot is verified and reused, not silently refreshed. To intentionally acquire another snapshot, use a different `STOCK_ML_ARTIFACTS` directory; changing snapshots after inspecting results does not restore holdout independence.

All OHLC fields use `auto_adjust=True`. This supports a consistent corporate-action-adjusted fractional-unit simulation, not literal historical share counts or a point-in-time execution database. VIX volume may be zero. Stocks must have positive volume. VIX-only dates outside SPY's equity calendar are excluded and recorded in metadata. Missing equity-session observations or invalid prices stop the workflow rather than being filled.

In [ ]:
from stockml.data import prepare_data
AS_OF = None  # Optional explicit completed calendar date, e.g. '2026-10-07'.
metadata = prepare_data(as_of=AS_OF)
display(pd.Series({key: metadata[key] for key in ['downloaded_at_utc', 'last_completed_session', 'synthetic', 'adjustment', 'excluded_vix_non_equity_dates']}))
display(pd.Series(metadata['versions'], name='installed_version'))

## Development-only exploratory analysis

Prices and returns can have changing volatility, heavy tails, and strong cross-stock dependence. Predictable volatility does not imply predictable return direction. The eight retrospectively chosen stocks also introduce survivorship and selection bias. Interpret the plots as descriptive evidence, not proof that a profitable pattern exists.

In [ ]:
from stockml.reporting import plot_eda
plot_eda()

## Feature and target definitions

All features use observations available at the signal close. Returns are fractions (0.01 means 1%). Rolling standard deviations use `ddof=1`. RSI uses Wilder-style exponential smoothing with `alpha=1/14`, `adjust=False`, and a 14-observation warm-up. MACD is the 12-span minus 26-span exponential moving average, divided by the current close. No scaling or imputation is fitted here.

| Category | Predictors |
|---|---|
| Momentum | 1-, 5-, 10-, 20-session returns |
| Trend | Close / 5- and 20-session mean close minus one |
| Volatility | 5- and 20-session standard deviation of daily returns |
| Volume | Volume percentage change; volume / 20-session mean volume |
| Technical | RSI-14; normalized MACD |
| Market context | SPY 1- and 5-session returns; VIX level and percentage change |
| Relative performance | Stock 5-session return minus SPY 5-session return |

The return target is $P_{t+5}/P_t-1$. Future risk is the sample standard deviation of returns on sessions $t+1,\ldots,t+5$. Both require all five future observations. Keep a `target_end` date for leakage checks. High-risk thresholds are fitted in Notebook 2, never here.

Rolling features are calculated continuously across year boundaries. Historical outcomes extending into 2026 are blanked before export. Inference rows with unknown future outcomes remain available.

In [ ]:
from stockml.common import FEATURES, read_table
features = read_table('history_features')
outcomes = read_table('history_outcomes')
assert len(FEATURES) == 17
assert not features.duplicated(['date', 'ticker']).any()
assert outcomes.target_end.dropna().max() < pd.Timestamp('2026-01-01')
display(features[features.date < '2024-01-01'].head())
display(outcomes[outcomes.date < '2024-01-01'].head())
print('Saved historical and holdout features, outcomes, and prices separately; SHA-256 fingerprints recorded.')

## Handoff

`data/history_{features,outcomes,prices}.parquet` contains the historical experiment inputs. `data/holdout_{features,outcomes,prices}.parquet` contains 2026 inputs. `data/metadata.json` records the snapshot, definitions, versions, and fingerprints.

Do not inspect holdout outcomes to modify features. Continue with Notebook 2.